# LeetCode 1273 — Delete Tree Nodes bằng Python

Link đề: https://leetcode.com/problems/delete-tree-nodes/ · Độ khó: **Medium** (Premium 🔒) · Chủ đề: Array, Tree, Depth-First Search, Breadth-First Search, DP on Trees

> **Ghi chú Premium:** đây là bài Premium. Đề bài, ví dụ và chữ ký hàm trong notebook được viết theo hiểu biết về bài, lấy từ trang doocs (https://leetcode.doocs.org/en/lc/1273/). Nếu bạn có tài khoản Premium, hãy đối chiếu lại với đề gốc trên LeetCode.

Skeleton LeetCode (theo lời giải Python của doocs):

```python
class Solution:
    def deleteTreeNodes(self, nodes: int, parent: list[int], value: list[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Biết cách **dựng cây từ mảng `parent`** thành danh sách con `children[p]` (danh sách kề) — kỹ năng gặp rất nhiều trong bài cây/đồ thị.
2. Hiểu tư duy **DFS hậu thứ tự (post-order)**: con xử lý xong rồi cha mới quyết định — mỗi lời gọi trả về một **cặp thông tin** `(tổng, số nút còn lại)`.
3. Đi từ lời giải **brute force `O(n²)`** (tính tổng cây con riêng cho từng nút) lên **`O(n)`** (một lần duyệt, gom thông tin từ dưới lên).
4. Viết được phiên bản **không đệ quy**: BFS lấy thứ tự rồi duyệt ngược, hoặc stack tường minh — tránh lỗi tràn đệ quy với cây dạng chuỗi dài `10^4`.
5. Nhận ra bẫy: **không được giả định `parent[i] < i`**, và khi một cây con bị xoá thì **mọi nút trong nó** biến mất, kể cả các nút có tổng khác 0.

## 2. Đề bài & ví dụ

Cho một cây gốc tại nút `0`, mô tả bởi:

- `nodes`: số nút (đánh số `0 … nodes-1`);
- `value[i]`: giá trị của nút `i` (có thể âm);
- `parent[i]`: cha của nút `i`, với `parent[0] = -1` (nút 0 là gốc).

Hãy **xoá mọi cây con có tổng giá trị bằng 0** (xoá cả cây con: nút đó và toàn bộ con cháu). Trả về **số nút còn lại** trong cây.

**Ví dụ 1:** `nodes = 7, parent = [-1,0,0,1,2,2,2], value = [1,-2,4,0,-2,-1,-1]` → `2`

**Ví dụ 2:** `nodes = 7, parent = [-1,0,0,1,2,2,2], value = [1,-2,4,0,-2,-1,-2]` → `6`

**Ràng buộc:**
- `1 <= nodes <= 10^4`
- `parent.length == nodes`, `value.length == nodes`
- `0 <= parent[i] <= nodes - 1` (với `i >= 1`), `parent[0] == -1`
- `-10^5 <= value[i] <= 10^5`
- Dữ liệu đảm bảo là một cây hợp lệ.

Lưu ý: ràng buộc **không nói** `parent[i] < i`. Cha có thể mang chỉ số lớn hơn con.

In [1]:
import random
import sys
import time
from collections import defaultdict, deque
from typing import List

examples = [
    (7, [-1, 0, 0, 1, 2, 2, 2], [1, -2, 4, 0, -2, -1, -1], 2),
    (7, [-1, 0, 0, 1, 2, 2, 2], [1, -2, 4, 0, -2, -1, -2], 6),
]
for nodes, parent, value, want in examples:
    print(f"nodes={nodes}, parent={parent}, value={value} -> mong đợi {want}")

nodes=7, parent=[-1, 0, 0, 1, 2, 2, 2], value=[1, -2, 4, 0, -2, -1, -1] -> mong đợi 2
nodes=7, parent=[-1, 0, 0, 1, 2, 2, 2], value=[1, -2, 4, 0, -2, -1, -2] -> mong đợi 6


## 3. Kiến thức nền

### 3.1 Từ mảng `parent` sang danh sách con

Mảng `parent` cho biết "ai là cha của tôi", nhưng để đi **từ trên xuống** ta cần biết "con của tôi là ai". Duyệt `i = 1 … n-1`, thêm `i` vào `children[parent[i]]`.

In [2]:
parent = [-1, 0, 0, 1, 2, 2, 2]
children = [[] for _ in range(len(parent))]
for i in range(1, len(parent)):
    children[parent[i]].append(i)
for p, ch in enumerate(children):
    print(f"nút {p}: con = {ch}")

nút 0: con = [1, 2]
nút 1: con = [3]
nút 2: con = [4, 5, 6]
nút 3: con = []
nút 4: con = []
nút 5: con = []
nút 6: con = []


### 3.2 `defaultdict(list)`

`defaultdict(list)` tự tạo danh sách rỗng khi truy cập khoá chưa có — tiện khi không muốn khởi tạo trước `n` danh sách. Lời giải doocs dùng đúng cách này.

In [3]:
g = defaultdict(list)
for i in range(1, len(parent)):
    g[parent[i]].append(i)
print(dict(g))
print("nút lá 5 có con:", g[5])   # truy cập khoá chưa có -> []

{0: [1, 2], 1: [3], 2: [4, 5, 6]}
nút lá 5 có con: []


### 3.3 Hàm đệ quy trả về nhiều giá trị (tuple)

Python cho phép `return (s, m)` rồi nhận bằng `t, n = f(...)`. Trong bài này mỗi nút cần báo lên cha **hai** thông tin: tổng cây con và số nút còn sống.

In [4]:
def min_max(xs):
    return min(xs), max(xs)

lo, hi = min_max([5, -3, 9, 0])
print("min =", lo, "| max =", hi)

min = -3 | max = 9


### 3.4 Giới hạn đệ quy của Python

Mặc định Python chỉ cho khoảng 1000 tầng gọi hàm lồng nhau. Cây dạng **chuỗi** `0 → 1 → 2 → … → 9999` khiến DFS đệ quy sâu 10^4 tầng. Có hai hướng: tăng giới hạn bằng `sys.setrecursionlimit`, hoặc viết **không đệ quy**.

In [5]:
print("giới hạn đệ quy hiện tại:", sys.getrecursionlimit())

def depth(k):
    return 0 if k == 0 else 1 + depth(k - 1)

print("depth(500) =", depth(500))

giới hạn đệ quy hiện tại: 1000
depth(500) = 500


### 3.5 `deque` cho BFS

`collections.deque` hỗ trợ `append` và `popleft` trong `O(1)`. BFS từ gốc cho ta một **thứ tự theo tầng**: cha luôn xuất hiện trước con. Đảo ngược thứ tự đó ⇒ con luôn được xử lý trước cha.

In [6]:
order = []
q = deque([0])
while q:
    u = q.popleft()
    order.append(u)
    q.extend(children[u])
print("thứ tự BFS:", order)
print("đảo ngược :", order[::-1])

thứ tự BFS: [0, 1, 2, 3, 4, 5, 6]
đảo ngược : [6, 5, 4, 3, 2, 1, 0]


## 4. Ý tưởng nền tảng / trực giác

Vẽ cây của **Ví dụ 1** (giá trị trong ngoặc):

```
            0(1)
          /      \
       1(-2)      2(4)
        |       /   |   \
       3(0)  4(-2) 5(-1) 6(-1)
```

Tính **tổng cây con** từ dưới lên:

| nút | các nút trong cây con | tổng | xoá? |
|---|---|---|---|
| 3 | {3} | 0 | **xoá** |
| 4 | {4} | -2 | không |
| 5 | {5} | -1 | không |
| 6 | {6} | -1 | không |
| 1 | {1, 3} | -2 + 0 = -2 | không |
| 2 | {2, 4, 5, 6} | 4 - 2 - 1 - 1 = 0 | **xoá** (mất cả 4, 5, 6) |
| 0 | tất cả | 1 - 2 + 0 = -1 | không |

Nút còn lại: `0` và `1` ⇒ **2** nút. Ở Ví dụ 2, `value[6] = -2` nên tổng cây con của nút 2 là `-1 ≠ 0` ⇒ chỉ nút 3 bị xoá ⇒ còn **6**.

Quan sát quan trọng: xoá một cây con có tổng 0 **không làm thay đổi tổng** của các tổ tiên (vì ta bỏ đi đúng một lượng bằng 0). Vậy ta có thể tính tổng cây con trên cây gốc rồi quyết định.

In [7]:
# Tính tay bằng code: tổng cây con của từng nút trong Ví dụ 1
nodes, parent, value, _ = examples[0]
children = [[] for _ in range(nodes)]
for i in range(1, nodes):
    children[parent[i]].append(i)

def subtree_nodes(u):
    res = [u]
    for v in children[u]:
        res += subtree_nodes(v)
    return res

for u in range(nodes):
    s = subtree_nodes(u)
    print(f"nút {u}: cây con {s}, tổng = {sum(value[x] for x in s)}")

nút 0: cây con [0, 1, 3, 2, 4, 5, 6], tổng = -1
nút 1: cây con [1, 3], tổng = -2
nút 2: cây con [2, 4, 5, 6], tổng = 0
nút 3: cây con [3], tổng = 0
nút 4: cây con [4], tổng = -2
nút 5: cây con [5], tổng = -1
nút 6: cây con [6], tổng = -1


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** làm đúng như định nghĩa.

1. Với mỗi nút `u`, duyệt toàn bộ cây con để tính `subsum[u]`.
2. Một nút `x` **bị xoá** nếu tồn tại một nút trên đường từ `x` lên gốc (kể cả `x`) có `subsum = 0`.
3. Đếm số nút không bị xoá.

Bước 2 có thể làm bằng cách đi ngược `parent` từ `x` lên gốc.

In [8]:
def delete_tree_nodes_brute(nodes, parent, value):
    children = [[] for _ in range(nodes)]
    for i in range(1, nodes):
        children[parent[i]].append(i)

    def subtree_sum(u):            # duyệt riêng cây con của u (dùng stack)
        total, st = 0, [u]
        while st:
            x = st.pop()
            total += value[x]
            st.extend(children[x])
        return total

    subsum = [subtree_sum(u) for u in range(nodes)]   # O(n) mỗi nút -> O(n^2)

    alive = 0
    for x in range(nodes):
        y, deleted = x, False
        while y != -1:                 # đi từ x lên gốc
            if subsum[y] == 0:
                deleted = True
                break
            y = parent[y]
        alive += not deleted
    return alive

for nodes, parent, value, want in examples:
    print(delete_tree_nodes_brute(nodes, parent, value), "| mong đợi", want)

2 | mong đợi 2
6 | mong đợi 6


**Độ phức tạp:** tính `subsum` cho mỗi nút tốn `O(kích thước cây con)`, tổng cộng `O(n²)` trong trường hợp xấu (cây dạng chuỗi). Bước kiểm tra tổ tiên cũng `O(n · chiều cao)` = `O(n²)`. Bộ nhớ `O(n)`.

**Nút thắt:** cùng một nút bị cộng lại nhiều lần — nút sâu nhất trong chuỗi được cộng vào tổng của **mọi** tổ tiên của nó. Với `n = 10^4` thì `~5·10^7` phép cộng, chậm trong Python. Ta cần **tái sử dụng** tổng của con khi tính tổng của cha.

In [9]:
# Đo nhanh trên chuỗi dài 2000 để thấy brute force tăng bậc hai
n = 2000
chain_parent = [-1] + list(range(n - 1))
chain_value = [1] * n
t = time.perf_counter()
print(delete_tree_nodes_brute(n, chain_parent, chain_value))
print(f"brute force chuỗi n={n}: {time.perf_counter() - t:.3f}s")

2000
brute force chuỗi n=2000: 0.299s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** tổng cây con của `u` = `value[u]` + tổng các cây con của các con. Số nút còn lại trong cây con của `u` cũng gom được từ các con:

```
sum(u)   = value[u] + Σ sum(v)        (v là con của u)
count(u) = 1        + Σ count(v)
nếu sum(u) == 0 thì count(u) = 0      (xoá cả cây con)
```

**Quy tắc "nếu… thì…":**
- Nếu một cây con có tổng 0 ⇒ báo lên cha "tôi còn 0 nút", nhưng **tổng vẫn là 0** (nên cộng vào cha không đổi gì).
- Nếu một con đã báo `count = 0` ⇒ cha không cần biết chi tiết bên dưới; chỉ cần cộng `0`.
- Nếu tổ tiên bị xoá ⇒ `count` của tổ tiên thành 0, tự động "nuốt" luôn các nút còn sống bên dưới.

**Invariant:** khi nút `u` được xử lý xong, `sum(u)` đúng bằng tổng giá trị cây con gốc `u` trên cây ban đầu, và `count(u)` đúng bằng số nút còn lại của cây con đó sau khi xoá (chỉ xét các xoá xảy ra bên trong cây con này).

Điều kiện cần: xử lý **con trước cha** ⇒ DFS hậu thứ tự, hoặc bất kỳ thứ tự nào mà con đứng trước cha (đảo ngược BFS).

## 7. Các cách tiếp theo

### Cách 2 — DFS đệ quy hậu thứ tự (lời giải doocs)

**Từng đoạn:**
1. Dựng `g` = danh sách con bằng `defaultdict(list)`.
2. `dfs(i)` khởi tạo `s = value[i]`, `m = 1` (chính nút `i`).
3. Gọi `dfs(j)` cho từng con `j`, cộng dồn tổng `t` và số nút `n`.
4. Nếu `s == 0` ⇒ `m = 0`. Trả `(s, m)`.
5. Đáp án là `dfs(0)[1]`.

In [10]:
def delete_tree_nodes_dfs(nodes, parent, value):
    g = defaultdict(list)
    for i in range(1, nodes):
        g[parent[i]].append(i)

    def dfs(i):
        s, m = value[i], 1
        for j in g[i]:
            t, n = dfs(j)
            s += t
            m += n
        if s == 0:
            m = 0
        return s, m

    return dfs(0)[1]

for nodes, parent, value, want in examples:
    print(delete_tree_nodes_dfs(nodes, parent, value), "| mong đợi", want)

2 | mong đợi 2
6 | mong đợi 6


Code nộp LeetCode cho Cách 2 (tăng giới hạn đệ quy để an toàn với chuỗi dài `10^4`):

In [11]:
class SolutionDFS:
    def deleteTreeNodes(self, nodes: int, parent: List[int], value: List[int]) -> int:
        sys.setrecursionlimit(max(10000, 2 * nodes + 100))
        g = defaultdict(list)
        for i in range(1, nodes):
            g[parent[i]].append(i)

        def dfs(i):
            s, m = value[i], 1
            for j in g[i]:
                t, n = dfs(j)
                s += t
                m += n
            if s == 0:
                m = 0
            return s, m

        return dfs(0)[1]

print([SolutionDFS().deleteTreeNodes(n, p, v) for n, p, v, _ in examples])

[2, 6]


**Độ phức tạp Cách 2:** mỗi nút được gọi `dfs` đúng một lần, mỗi cạnh duyệt một lần ⇒ `O(n)` thời gian. Bộ nhớ `O(n)` cho `g` và ngăn xếp đệ quy (sâu tới `O(n)` khi cây là chuỗi).

### Cách 3 — BFS lấy thứ tự, rồi duyệt ngược (không đệ quy)

**Ý tưởng:** BFS từ gốc cho thứ tự trong đó cha luôn đứng trước con. Đi **ngược** thứ tự này, khi gặp nút `u` thì mọi con của `u` đã được xử lý xong. Ta cộng thông tin của `u` lên `parent[u]`.

**Từng đoạn:**
1. `s[i] = value[i]`, `cnt[i] = 1` cho mọi nút.
2. Lấy `order` bằng BFS.
3. Với `u` trong `reversed(order)`: nếu `s[u] == 0` thì `cnt[u] = 0`; sau đó nếu `u != 0` thì `s[parent[u]] += s[u]`, `cnt[parent[u]] += cnt[u]`.
4. Trả `cnt[0]`.

In [12]:
def delete_tree_nodes_bfs(nodes, parent, value):
    children = [[] for _ in range(nodes)]
    for i in range(1, nodes):
        children[parent[i]].append(i)

    order, q = [], deque([0])
    while q:
        u = q.popleft()
        order.append(u)
        q.extend(children[u])

    s = value[:]                 # bản sao, không sửa input
    cnt = [1] * nodes
    for u in reversed(order):    # con luôn được xử lý trước cha
        if s[u] == 0:
            cnt[u] = 0
        if u != 0:
            s[parent[u]] += s[u]
            cnt[parent[u]] += cnt[u]
    return cnt[0]

for nodes, parent, value, want in examples:
    print(delete_tree_nodes_bfs(nodes, parent, value), "| mong đợi", want)

2 | mong đợi 2
6 | mong đợi 6


In [13]:
class SolutionBFS:
    def deleteTreeNodes(self, nodes: int, parent: List[int], value: List[int]) -> int:
        children = [[] for _ in range(nodes)]
        for i in range(1, nodes):
            children[parent[i]].append(i)
        order, q = [], deque([0])
        while q:
            u = q.popleft()
            order.append(u)
            q.extend(children[u])
        s, cnt = value[:], [1] * nodes
        for u in reversed(order):
            if s[u] == 0:
                cnt[u] = 0
            if u:
                s[parent[u]] += s[u]
                cnt[parent[u]] += cnt[u]
        return cnt[0]

print([SolutionBFS().deleteTreeNodes(n, p, v) for n, p, v, _ in examples])

[2, 6]


**Độ phức tạp Cách 3:** `O(n)` thời gian (BFS + một lượt ngược), `O(n)` bộ nhớ. Ưu điểm: **không có đệ quy** ⇒ không lo giới hạn đệ quy.

### Cách 4 — Stack tường minh mô phỏng hậu thứ tự

**Ý tưởng:** đẩy `(nút, đã_mở?)` vào stack. Lần đầu lấy ra (`False`): đẩy lại `(nút, True)` rồi đẩy các con. Lần thứ hai (`True`): các con đã xong, gom thông tin. Đây chính là DFS đệ quy được "trải phẳng" (giống Solution 2 trên doocs).

In [14]:
def delete_tree_nodes_stack(nodes, parent, value):
    g = [[] for _ in range(nodes)]
    for i in range(1, nodes):
        g[parent[i]].append(i)

    s = [0] * nodes
    m = [0] * nodes
    st = [(0, False)]
    while st:
        i, done = st.pop()
        if not done:
            st.append((i, True))          # hẹn quay lại sau khi xong các con
            for j in g[i]:
                st.append((j, False))
        else:
            s[i], m[i] = value[i], 1
            for j in g[i]:
                s[i] += s[j]
                m[i] += m[j]
            if s[i] == 0:
                m[i] = 0
    return m[0]

for nodes, parent, value, want in examples:
    print(delete_tree_nodes_stack(nodes, parent, value), "| mong đợi", want)

2 | mong đợi 2
6 | mong đợi 6


**Độ phức tạp Cách 4:** mỗi nút vào stack hai lần ⇒ `O(n)` thời gian, `O(n)` bộ nhớ.

### Cách 5 (có điều kiện) — Duyệt ngược chỉ số nếu `parent[i] < i`

Nếu dữ liệu đảm bảo **mọi cha có chỉ số nhỏ hơn con** (`parent[i] < i`), thì duyệt `i = n-1 … 1` đã là thứ tự "con trước cha", không cần dựng `children`. Nhưng ràng buộc của bài chỉ nói `0 <= parent[i] <= nodes - 1`, **không** đảm bảo điều này. Vì vậy ta kiểm tra điều kiện trước, nếu không thoả thì quay về Cách 3.

In [15]:
def delete_tree_nodes_reverse_index(nodes, parent, value):
    if any(parent[i] >= i for i in range(1, nodes)):
        return delete_tree_nodes_bfs(nodes, parent, value)   # không an toàn -> dùng BFS
    s, cnt = value[:], [1] * nodes
    for i in range(nodes - 1, -1, -1):
        if s[i] == 0:
            cnt[i] = 0
        if i:
            s[parent[i]] += s[i]
            cnt[parent[i]] += cnt[i]
    return cnt[0]

for nodes, parent, value, want in examples:
    print(delete_tree_nodes_reverse_index(nodes, parent, value), "| mong đợi", want)

2 | mong đợi 2
6 | mong đợi 6


## 8. TRACE từng bước

Trace Cách 3 trên **Ví dụ 1**. Thứ tự BFS là `[0, 1, 2, 3, 4, 5, 6]`, duyệt ngược `[6, 5, 4, 3, 2, 1, 0]`.

| bước | u | s[u] | cnt[u] sau kiểm tra | cộng lên cha | s[cha] | cnt[cha] |
|---|---|---|---|---|---|---|
| 1 | 6 | -1 | 1 | 2 | 4-1=3 | 2 |
| 2 | 5 | -1 | 1 | 2 | 2 | 3 |
| 3 | 4 | -2 | 1 | 2 | 0 | 4 |
| 4 | 3 | 0 | **0** | 1 | -2 | 1 |
| 5 | 2 | 0 | **0** | 0 | 1+0=1 | 1+0=1 |
| 6 | 1 | -2 | 1 | 0 | -1 | 2 |
| 7 | 0 | -1 | 2 | — | — | — |

Kết quả `cnt[0] = 2`.

In [16]:
def trace_bfs(nodes, parent, value):
    children = [[] for _ in range(nodes)]
    for i in range(1, nodes):
        children[parent[i]].append(i)
    order, q = [], deque([0])
    while q:
        u = q.popleft()
        order.append(u)
        q.extend(children[u])
    print("BFS order:", order)
    s, cnt = value[:], [1] * nodes
    for step, u in enumerate(reversed(order), 1):
        if s[u] == 0:
            cnt[u] = 0
        msg = f"bước {step}: u={u}, s[u]={s[u]:>3}, cnt[u]={cnt[u]}"
        if u != 0:
            p = parent[u]
            s[p] += s[u]
            cnt[p] += cnt[u]
            msg += f" -> cha {p}: s={s[p]:>3}, cnt={cnt[p]}"
        print(msg)
    print("đáp án =", cnt[0])
    return cnt[0]

trace_bfs(*examples[0][:3])

BFS order: [0, 1, 2, 3, 4, 5, 6]
bước 1: u=6, s[u]= -1, cnt[u]=1 -> cha 2: s=  3, cnt=2
bước 2: u=5, s[u]= -1, cnt[u]=1 -> cha 2: s=  2, cnt=3
bước 3: u=4, s[u]= -2, cnt[u]=1 -> cha 2: s=  0, cnt=4
bước 4: u=3, s[u]=  0, cnt[u]=0 -> cha 1: s= -2, cnt=1
bước 5: u=2, s[u]=  0, cnt[u]=0 -> cha 0: s=  1, cnt=1
bước 6: u=1, s[u]= -2, cnt[u]=1 -> cha 0: s= -1, cnt=2
bước 7: u=0, s[u]= -1, cnt[u]=2
đáp án = 2


2

Trace Cách 2 (DFS đệ quy) với thụt lề theo độ sâu để thấy rõ thứ tự **vào** và **ra** của mỗi nút:

In [17]:
def trace_dfs(nodes, parent, value):
    g = defaultdict(list)
    for i in range(1, nodes):
        g[parent[i]].append(i)

    def dfs(i, d):
        print("  " * d + f"vào nút {i} (value={value[i]})")
        s, m = value[i], 1
        for j in g[i]:
            t, n = dfs(j, d + 1)
            s += t
            m += n
        if s == 0:
            m = 0
        print("  " * d + f"ra  nút {i}: tổng={s}, còn={m}" + ("  <- XOÁ" if s == 0 else ""))
        return s, m

    return dfs(0, 0)[1]

print("đáp án =", trace_dfs(*examples[1][:3]))

vào nút 0 (value=1)
  vào nút 1 (value=-2)
    vào nút 3 (value=0)
    ra  nút 3: tổng=0, còn=0  <- XOÁ
  ra  nút 1: tổng=-2, còn=1
  vào nút 2 (value=4)
    vào nút 4 (value=-2)
    ra  nút 4: tổng=-2, còn=1
    vào nút 5 (value=-1)
    ra  nút 5: tổng=-1, còn=1
    vào nút 6 (value=-2)
    ra  nút 6: tổng=-2, còn=1
  ra  nút 2: tổng=-1, còn=4
ra  nút 0: tổng=-2, còn=6
đáp án = 6


## 9. Code nộp LeetCode

Chọn Cách 3 (BFS + duyệt ngược): `O(n)`, không đệ quy, không giả định thứ tự chỉ số. Chữ ký đúng theo doocs: `deleteTreeNodes(self, nodes, parent, value)`.

In [18]:
class Solution:
    def deleteTreeNodes(self, nodes: int, parent: List[int], value: List[int]) -> int:
        children = [[] for _ in range(nodes)]
        for i in range(1, nodes):
            children[parent[i]].append(i)

        order = [0]
        for u in order:                 # BFS: duyệt danh sách đang lớn dần
            order.extend(children[u])

        s = value[:]
        cnt = [1] * nodes
        for u in reversed(order):
            if s[u] == 0:
                cnt[u] = 0
            if u != 0:
                s[parent[u]] += s[u]
                cnt[parent[u]] += cnt[u]
        return cnt[0]

for nodes, parent, value, want in examples:
    got = Solution().deleteTreeNodes(nodes, parent, value)
    print(got, "OK" if got == want else "SAI")

2 OK
6 OK


Mẹo nhỏ trong code nộp: `for u in order: order.extend(children[u])` — vòng `for` trên list Python đọc theo chỉ số, nên phần tử được thêm vào cuối trong lúc lặp vẫn được duyệt. Đây là một cách viết BFS gọn không cần `deque`.

## 10. Vì sao thuật toán đúng

1. **Thứ tự xử lý:** BFS thêm con vào sau cha, nên trong `order` mọi nút đứng sau cha của nó. Duyệt ngược ⇒ mọi con được xử lý **trước** cha.
2. **Tổng đúng (quy nạp từ lá lên):** lá có `s = value`. Giả sử mọi con `v` của `u` đã có `s[v]` = tổng cây con của `v`. Khi xử lý từng con, `s[u]` được cộng thêm `s[v]`, nên lúc tới lượt `u`, `s[u] = value[u] + Σ s[v]` = tổng cây con của `u`.
3. **Số nút đúng:** tương tự, `cnt[u] = 1 + Σ cnt[v]` là số nút còn lại khi chỉ xét các xoá bên trong các con; nếu `s[u] = 0` thì cả cây con bị xoá ⇒ `cnt[u] = 0`.
4. **Xoá không làm sai tổng tổ tiên:** một cây con bị xoá có tổng 0, nên việc vẫn cộng `s[v] = 0` vào cha tương đương với việc đã bỏ nó khỏi cây. Vì thế không cần "tính lại" tổng sau khi xoá.
5. **Không đếm trùng / sót:** mỗi nút đóng góp đúng `1` vào `cnt` của chính nó, và được truyền lên qua đúng một con đường tới gốc, bị triệt tiêu đúng khi có một tổ tiên (hoặc chính nó) có tổng 0.

## 11. Phân tích Time / Space

Gọi `n = nodes`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(n²)` worst (cây chuỗi) | `O(n)` | trung bình nhanh hơn nếu cây thấp |
| 2. DFS đệ quy | `O(n)` | `O(n)` (stack đệ quy tới `n`) | cần tăng recursion limit |
| 3. BFS + ngược | `O(n)` | `O(n)` | không đệ quy |
| 4. Stack tường minh | `O(n)` | `O(n)` | mỗi nút vào stack 2 lần |
| 5. Ngược chỉ số | `O(n)` | `O(n)` | chỉ khi `parent[i] < i` |

Với `n = 10^4`, các cách `O(n)` chạy trong vài mili-giây.

## 12. Bộ test

Gồm: ví dụ đề, edge case (1 nút, toàn 0, gốc có tổng 0, chuỗi dài), cây có `parent[i] > i`, và đối chiếu ngẫu nhiên giữa mọi cách (seed cố định).

In [19]:
ALL = {
    "brute": delete_tree_nodes_brute,
    "dfs": delete_tree_nodes_dfs,
    "bfs": delete_tree_nodes_bfs,
    "stack": delete_tree_nodes_stack,
    "rev": delete_tree_nodes_reverse_index,
    "Solution": lambda n, p, v: Solution().deleteTreeNodes(n, p, v),
}

tests = [
    (7, [-1, 0, 0, 1, 2, 2, 2], [1, -2, 4, 0, -2, -1, -1], 2),
    (7, [-1, 0, 0, 1, 2, 2, 2], [1, -2, 4, 0, -2, -1, -2], 6),
    (1, [-1], [0], 0),                       # một nút giá trị 0 -> xoá hết
    (1, [-1], [5], 1),
    (3, [-1, 0, 0], [0, 0, 0], 0),           # toàn 0: lá bị xoá, gốc cũng tổng 0
    (3, [-1, 0, 0], [2, -1, -1], 0),         # gốc tổng 0 -> xoá cả cây
    (3, [-1, 0, 1], [5, 3, -3], 1),          # nút 1 có tổng 0 -> mất cả 1 và 2
    (3, [-1, 2, 0], [1, -1, 1], 1),          # parent[1] = 2 > 1
    (4, [-1, 3, 3, 0], [1, 2, -2, 0], 1),    # cha chỉ số lớn hơn con
]
for nodes, parent, value, want in tests:
    for name, f in ALL.items():
        got = f(nodes, parent, value)
        assert got == want, (name, nodes, parent, value, got, want)
print("Tất cả test cố định đều qua:", len(tests), "test x", len(ALL), "cách")

Tất cả test cố định đều qua: 9 test x 6 cách


In [20]:
def random_tree(n, rng, lo=-3, hi=3):
    """Sinh cây ngẫu nhiên gốc 0, nhãn các nút còn lại bị xáo trộn (parent[i] có thể > i)."""
    perm = list(range(1, n))
    rng.shuffle(perm)
    labels = [0] + perm                  # labels[k] = nhãn của nút thứ k khi sinh
    parent = [-1] * n
    for k in range(1, n):
        parent[labels[k]] = labels[rng.randrange(k)]
    value = [rng.randint(lo, hi) for _ in range(n)]
    return parent, value

rng = random.Random(1273)
bad_order = 0
for _ in range(600):
    n = rng.randint(1, 12)
    parent, value = random_tree(n, rng)
    bad_order += any(parent[i] > i for i in range(1, n))
    want = delete_tree_nodes_brute(n, parent, value)
    for name, f in ALL.items():
        assert f(n, parent, value) == want, (name, parent, value)
print("600 cây ngẫu nhiên: mọi cách khớp brute force;", bad_order, "cây có parent[i] > i")

600 cây ngẫu nhiên: mọi cách khớp brute force; 416 cây có parent[i] > i


In [21]:
# Test cực trị: chuỗi dài 10^4 (đệ quy sâu) và cây sao 10^4 nút
n = 10_000
chain_parent = [-1] + list(range(n - 1))
chain_value = [rng.randint(-10**5, 10**5) for _ in range(n)]
star_parent = [-1] + [0] * (n - 1)
star_value = [rng.choice([-1, 0, 1]) for _ in range(n)]

for name in ["dfs", "bfs", "stack", "Solution"]:
    f = ALL[name]
    if name == "dfs":
        sys.setrecursionlimit(30000)
    t = time.perf_counter()
    a = f(n, chain_parent, chain_value)
    b = f(n, star_parent, star_value)
    print(f"{name:>8}: chuỗi -> {a}, sao -> {b}, {time.perf_counter() - t:.3f}s")

     dfs: chuỗi -> 10000, sao -> 6638, 0.011s
     bfs: chuỗi -> 10000, sao -> 6638, 0.011s
   stack: chuỗi -> 10000, sao -> 6638, 0.012s
Solution: chuỗi -> 10000, sao -> 6638, 0.005s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ đếm các nút có tổng cây con khác 0

Sai vì một nút có tổng ≠ 0 vẫn **bị xoá theo** nếu một tổ tiên của nó có tổng 0.

In [22]:
def wrong_count_nonzero(nodes, parent, value):
    children = [[] for _ in range(nodes)]
    for i in range(1, nodes):
        children[parent[i]].append(i)
    order = [0]
    for u in order:
        order.extend(children[u])
    s = value[:]
    for u in reversed(order):
        if u:
            s[parent[u]] += s[u]
    return sum(1 for x in s if x != 0)     # SAI: quên nút con của cây con bị xoá

n, p, v, want = examples[0]
print("Code sai trả về:", wrong_count_nonzero(n, p, v), "| đúng là:", want)
print("Nút 4, 5, 6 có tổng khác 0 nhưng cha 2 có tổng 0 nên cũng phải bị xoá.")

Code sai trả về: 5 | đúng là: 2
Nút 4, 5, 6 có tổng khác 0 nhưng cha 2 có tổng 0 nên cũng phải bị xoá.


**Code đúng:** truyền `cnt` từ dưới lên và đặt `cnt[u] = 0` khi `s[u] == 0` (như `Solution`), để việc xoá ở tổ tiên tự động "nuốt" cả cây con.

In [23]:
print("Code đúng:", Solution().deleteTreeNodes(n, p, v))

Code đúng: 2


### Lỗi 2 — Giả định `parent[i] < i` rồi duyệt ngược chỉ số

Ràng buộc chỉ cho `0 <= parent[i] <= nodes - 1`. Nếu cha mang chỉ số lớn hơn, khi duyệt `i` giảm dần ta sẽ cộng nút cha lên ông **trước** khi cha nhận đủ thông tin từ con.

In [24]:
def wrong_reverse_index(nodes, parent, value):
    s, cnt = value[:], [1] * nodes
    for i in range(nodes - 1, 0, -1):          # SAI nếu có parent[i] > i
        if s[i] == 0:
            cnt[i] = 0
        s[parent[i]] += s[i]
        cnt[parent[i]] += cnt[i]
    if s[0] == 0:
        cnt[0] = 0
    return cnt[0]

case = (3, [-1, 2, 0], [1, -1, 1])   # cây 0 -> 2 -> 1
print("Code sai trả về:", wrong_reverse_index(*case),
      "| đúng là:", delete_tree_nodes_brute(*case))

Code sai trả về: 2 | đúng là: 1


Ở cây `0 → 2 → 1`: nút 2 có tổng `1 + (-1) = 0` nên phải bị xoá cùng nút 1, còn lại 1 nút. Code sai xử lý `i = 2` trước `i = 1`, lúc đó `s[2] = 1 ≠ 0` nên cộng cả 2 nút lên gốc.

**Code đúng:** lấy thứ tự bằng BFS/DFS (Cách 3, 4) hoặc kiểm tra điều kiện trước như Cách 5.

In [25]:
print("Code đúng:", delete_tree_nodes_bfs(*case), "|", delete_tree_nodes_reverse_index(*case))

Code đúng: 1 | 1


### Lỗi 3 — DFS đệ quy không tăng giới hạn đệ quy

Trên cây dạng chuỗi dài, lời gọi đệ quy lồng nhau vượt quá giới hạn mặc định (~1000) ⇒ `RecursionError`. Ở đây ta mô phỏng giới hạn mặc định và **bắt** lỗi để in ra thay vì để chương trình dừng.

In [26]:
old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(1000)                     # mô phỏng môi trường mặc định
n = 3000
chain_parent = [-1] + list(range(n - 1))
chain_value = [1] * n
try:
    print(delete_tree_nodes_dfs(n, chain_parent, chain_value))
except RecursionError as e:
    print("Code sai gặp lỗi:", type(e).__name__, "-", str(e)[:45])
finally:
    sys.setrecursionlimit(old_limit)

Code sai gặp lỗi: RecursionError - maximum recursion depth exceeded


**Code đúng:** dùng phiên bản không đệ quy (Cách 3/4), hoặc gọi `sys.setrecursionlimit` đủ lớn trước khi DFS (như `SolutionDFS`).

In [27]:
print("Không đệ quy:", delete_tree_nodes_stack(n, chain_parent, chain_value))
print("DFS + tăng limit:", SolutionDFS().deleteTreeNodes(n, chain_parent, chain_value))

Không đệ quy: 3000
DFS + tăng limit: 3000


### Lỗi 4 — Cộng `cnt` lên cha **trước** khi kiểm tra tổng 0

Thứ tự hai dòng rất quan trọng: phải đặt `cnt[u] = 0` (nếu `s[u] == 0`) **rồi mới** cộng lên cha.

In [28]:
def wrong_order_of_lines(nodes, parent, value):
    children = [[] for _ in range(nodes)]
    for i in range(1, nodes):
        children[parent[i]].append(i)
    order = [0]
    for u in order:
        order.extend(children[u])
    s, cnt = value[:], [1] * nodes
    for u in reversed(order):
        if u:
            s[parent[u]] += s[u]
            cnt[parent[u]] += cnt[u]       # SAI: cộng trước
        if s[u] == 0:
            cnt[u] = 0                     # đặt 0 quá muộn, cha đã nhận số cũ
    return cnt[0]

n, p, v, want = examples[1]
print("Code sai trả về:", wrong_order_of_lines(n, p, v), "| đúng là:", want)

Code sai trả về: 7 | đúng là: 6


**Code đúng:** đảo lại thứ tự: kiểm tra `s[u] == 0` trước, cộng lên cha sau (xem `Solution` ở mục 9).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy **mảng `parent`** ⇒ việc đầu tiên: dựng `children`. Gốc là nút có `parent = -1`.
- Thấy câu hỏi về **"cây con"** (tổng, kích thước, max, có tính chất gì) ⇒ nghĩ ngay **DFS hậu thứ tự** trả về một bộ thông tin.
- Hàm DFS trả về **tuple** khi cần nhiều thông tin; quyết định "xoá/giữ" của cha dựa trên thông tin tổng hợp từ con.
- Python + cây có thể sâu ⇒ ưu tiên **BFS + duyệt ngược** hoặc **stack**, hoặc nhớ `sys.setrecursionlimit`.
- Đừng giả định chỉ số cha nhỏ hơn con trừ khi đề ghi rõ.
- Mẫu "tính trên cây gốc rồi mới xoá" đúng ở đây vì xoá phần có **tổng 0** không làm đổi tổng tổ tiên. Nếu điều kiện xoá khác (ví dụ tổng âm), phải dùng tổng **sau khi xoá** — xem biến thể.

## 15. Biến thể của bài

1. **Xoá cây con có tổng âm:** khi đó xoá làm **thay đổi** tổng của tổ tiên ⇒ con bị xoá phải báo lên tổng `0` (không phải tổng thật). Đây là ý tưởng của LC 814 *Binary Tree Pruning* (xoá cây con toàn số 0) và các bài "cắt tỉa cây".
2. **Trả về tổng giá trị còn lại** thay vì số nút.
3. **Trả về danh sách nút còn lại** (theo thứ tự tăng).
4. **Xoá cây con có tổng chia hết cho `k`** — tương tự, nhưng lúc này xoá có thể làm đổi tổng tổ tiên (tổng là bội của `k` nhưng khác 0).
5. Các bài cùng họ "DP trên cây từ mảng parent": LC 1519 *Number of Nodes in the Sub-Tree With the Same Label*, LC 2049 *Count Nodes With the Highest Score*, LC 1376 *Time Needed to Inform All Employees*.

In [29]:
def delete_negative_subtrees(nodes, parent, value):
    """Biến thể: xoá cây con có tổng < 0, tổng tính SAU khi đã xoá các cây con bên dưới."""
    children = [[] for _ in range(nodes)]
    for i in range(1, nodes):
        children[parent[i]].append(i)
    order = [0]
    for u in order:
        order.extend(children[u])
    s, cnt = value[:], [1] * nodes
    for u in reversed(order):
        if s[u] < 0:
            s[u], cnt[u] = 0, 0        # bị xoá: báo lên cha tổng 0
        if u:
            s[parent[u]] += s[u]
            cnt[parent[u]] += cnt[u]
    return cnt[0], s[0]

print(delete_negative_subtrees(*examples[0][:3]))   # (số nút còn, tổng còn)

(2, 5)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | tính tổng từng cây con riêng, kiểm tra tổ tiên | `O(n²)` | `O(n)` | kiểm chứng, `n` nhỏ |
| 2. DFS đệ quy | hậu thứ tự trả `(tổng, số nút)` | `O(n)` | `O(n)` | code ngắn nhất; nhớ tăng recursion limit |
| 3. BFS + duyệt ngược | thứ tự tầng đảo ngược = con trước cha | `O(n)` | `O(n)` | **khuyên dùng** trong Python |
| 4. Stack tường minh | mô phỏng đệ quy bằng `(nút, done)` | `O(n)` | `O(n)` | khi cần thứ tự hậu thứ tự thật sự |
| 5. Ngược chỉ số | `i` giảm dần khi `parent[i] < i` | `O(n)` | `O(n)` | chỉ khi đề bảo đảm điều kiện |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** Không chạy code, đoán kết quả:
- `deleteTreeNodes(4, [-1, 0, 1, 2], [3, -1, -1, -1])`
- `deleteTreeNodes(4, [-1, 0, 0, 0], [0, 1, -1, 0])`

**Bài 2 (tính tay):** Với `nodes = 6, parent = [-1, 0, 0, 1, 1, 2], value = [2, 1, -3, -1, 0, 3]`, hãy lập bảng tổng cây con từng nút, đánh dấu nút bị xoá, và cho biết số nút còn lại.

**Bài 3 (điền TODO):** Hoàn thiện hàm `subtree_sums_todo` trả về mảng tổng cây con của mọi nút (không đệ quy).

**Bài 4 (tự viết lại):** Viết `remaining_nodes(nodes, parent, value)` trả về **danh sách các nút còn lại** (tăng dần) sau khi xoá.

**Bài 5 (mini challenge):** Viết `delete_div_k(nodes, parent, value, k)`: lặp lại việc xoá cây con có tổng (sau khi đã xoá bên dưới) **chia hết cho `k`**; trả về số nút còn lại. Lưu ý gốc cũng có thể bị xoá.

In [30]:
def subtree_sums_todo(nodes, parent, value):
    children = [[] for _ in range(nodes)]
    for i in range(1, nodes):
        children[parent[i]].append(i)
    order = [0]
    # TODO 1: mở rộng order theo BFS
    s = value[:]
    # TODO 2: duyệt ngược order, cộng s[u] lên s[parent[u]]
    return s

# Lưu ý: cell này chỉ định nghĩa hàm, không gọi.

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Vẽ cây ra giấy. (2) Ở cây thứ nhất là chuỗi `0→1→2→3`: tổng từ dưới lên là `-1, -2, -3, 0`. (3) Gốc có tổng 0 thì còn bao nhiêu nút? Ở cây thứ hai, chú ý gốc có giá trị 0 nhưng tổng cây con của gốc là `0 + 1 - 1 + 0`.

**Bài 2.** (1) Dựng `children`: `0:[1,2]`, `1:[3,4]`, `2:[5]`. (2) Tính lá trước: `3:-1`, `4:0`, `5:3`. (3) Rồi `1 = 1 - 1 + 0 = 0`, `2 = -3 + 3 = 0`, `0 = 2 + 0 + 0`.

**Bài 3.** (1) `for u in order: order.extend(children[u])`. (2) `for u in reversed(order): if u: s[parent[u]] += s[u]`. (3) So với brute force tổng cây con ở mục 4.

**Bài 4.** (1) Một nút còn lại khi không có tổ tiên nào (kể cả chính nó) có tổng 0. (2) Duyệt BFS **xuôi**: `dead[u] = dead[parent[u]] or s[u] == 0`. (3) Lấy các `u` có `dead[u] == False`, sort.

**Bài 5.** (1) Giống biến thể "tổng âm": bị xoá thì báo lên cha tổng 0, số nút 0. (2) Điều kiện xoá: `s[u] % k == 0` (Python `%` cho số âm vẫn trả 0..k-1). (3) Kiểm chứng bằng một brute force đệ quy trên cây nhỏ.

## 19. Đáp án tham khảo

In [31]:
# Bài 1
assert Solution().deleteTreeNodes(4, [-1, 0, 1, 2], [3, -1, -1, -1]) == 0
assert Solution().deleteTreeNodes(4, [-1, 0, 0, 0], [0, 1, -1, 0]) == 0
# Bài 2: tổng cây con 0 -> 2, 1 -> 0 (xoá 1,3,4), 2 -> 0 (xoá 2,5) => chỉ còn nút 0
assert Solution().deleteTreeNodes(6, [-1, 0, 0, 1, 1, 2], [2, 1, -3, -1, 0, 3]) == 1

# Bài 3
def subtree_sums_sol(nodes, parent, value):
    children = [[] for _ in range(nodes)]
    for i in range(1, nodes):
        children[parent[i]].append(i)
    order = [0]
    for u in order:
        order.extend(children[u])
    s = value[:]
    for u in reversed(order):
        if u:
            s[parent[u]] += s[u]
    return s

assert subtree_sums_sol(6, [-1, 0, 0, 1, 1, 2], [2, 1, -3, -1, 0, 3]) == [2, 0, 0, -1, 0, 3]

# Bài 4
def remaining_nodes_sol(nodes, parent, value):
    s = subtree_sums_sol(nodes, parent, value)
    children = [[] for _ in range(nodes)]
    for i in range(1, nodes):
        children[parent[i]].append(i)
    dead = [False] * nodes
    order = [0]
    for u in order:
        dead[u] = (s[u] == 0) or (u != 0 and dead[parent[u]])
        order.extend(children[u])
    return [u for u in range(nodes) if not dead[u]]

assert remaining_nodes_sol(*examples[0][:3]) == [0, 1]
assert remaining_nodes_sol(*examples[1][:3]) == [0, 1, 2, 4, 5, 6]

# Bài 5
def delete_div_k_sol(nodes, parent, value, k):
    children = [[] for _ in range(nodes)]
    for i in range(1, nodes):
        children[parent[i]].append(i)
    order = [0]
    for u in order:
        order.extend(children[u])
    s, cnt = value[:], [1] * nodes
    for u in reversed(order):
        if s[u] % k == 0:
            s[u], cnt[u] = 0, 0
        if u:
            s[parent[u]] += s[u]
            cnt[parent[u]] += cnt[u]
    return cnt[0]

def delete_div_k_brute(nodes, parent, value, k):
    children = [[] for _ in range(nodes)]
    for i in range(1, nodes):
        children[parent[i]].append(i)
    def go(u):                       # trả (tổng còn lại, số nút còn lại)
        tot, c = value[u], 1
        for v in children[u]:
            a, b = go(v)
            tot += a
            c += b
        return (0, 0) if tot % k == 0 else (tot, c)
    return go(0)[1]

rng = random.Random(99)
for _ in range(400):
    n = rng.randint(1, 10)
    par, val = random_tree(n, rng, -4, 4)
    want = delete_tree_nodes_brute(n, par, val)
    assert remaining_nodes_sol(n, par, val).__len__() == want
    assert subtree_sums_sol(n, par, val)[0] == sum(val)
    k = rng.randint(1, 5)
    assert delete_div_k_sol(n, par, val, k) == delete_div_k_brute(n, par, val, k)
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "DP trên cây từ mảng parent" (không đệ quy):**

```python
children = [[] for _ in range(n)]
for i in range(1, n):
    children[parent[i]].append(i)
order = [root]
for u in order:                 # BFS: cha trước con
    order.extend(children[u])
for u in reversed(order):       # con trước cha
    # 1) chốt thông tin của u (dựa trên con đã xong)
    # 2) đẩy thông tin của u lên parent[u]
    ...
```

**Template DFS đệ quy trả tuple:**

```python
def dfs(u):
    s, m = value[u], 1
    for v in children[u]:
        t, k = dfs(v); s += t; m += k
    if s == 0: m = 0
    return s, m
```

**Checklist trước khi submit:**
- [ ] Dựng `children` từ `i = 1` (bỏ gốc có `parent = -1`).
- [ ] Không giả định `parent[i] < i`.
- [ ] Kiểm tra `s[u] == 0` **trước** khi cộng `cnt[u]` lên cha.
- [ ] Gốc có tổng 0 ⇒ đáp án 0.
- [ ] Nếu dùng đệ quy: `sys.setrecursionlimit` đủ lớn cho chuỗi `10^4`.
- [ ] Không sửa trực tiếp `value` nếu còn cần (dùng `value[:]`).
- [ ] Đúng chữ ký: `deleteTreeNodes(self, nodes: int, parent: List[int], value: List[int]) -> int`.